# Data Quality Infrastructure Setup

**Purpose**: One-time setup for data quality framework  
**Run**: Once at project start (idempotent - safe to re-run)  
**Creates**:
1. `gold.data_quality_metrics` - Central logging table for all quality checks
2. `gold.dim_customers_quarantine` - Failed customer records
3. `gold.dim_products_quarantine` - Failed product records
4. `gold.fact_sales_quarantine` - Failed sales fact records

**Retail Analytics Best Practice #2**: Implement data quality checks with quarantine pattern

## Quarantine Pattern Benefits
- **Non-blocking**: Pipeline continues even with quality issues
- **Auditable**: Full history of failures and remediations
- **Investigatable**: Original data preserved for debugging
- **Remediable**: Track fixes and re-process corrected data

## After Setup
Use `data_quality_checks` notebook with parameters to run checks on any table.

In [0]:
%sql
-- Central metrics table for all data quality checks
CREATE TABLE IF NOT EXISTS workspace.gold.data_quality_metrics (
  table_name STRING,
  check_timestamp TIMESTAMP,
  total_records BIGINT,
  passed_records BIGINT,
  failed_records BIGINT,
  critical_issues BIGINT,
  error_issues BIGINT,
  warning_issues BIGINT,
  completeness_rate DOUBLE,
  validity_rate DOUBLE,
  run_duration_seconds DOUBLE,
  check_details STRING  -- JSON with detailed breakdowns
)
USING DELTA;

SELECT 'Metrics table created' as status;

In [0]:
%sql
-- Quarantine table for dim_customers failed quality checks
CREATE TABLE IF NOT EXISTS workspace.gold.dim_customers_quarantine (
  -- Original customer columns
  customer_key BIGINT,
  customer_id STRING,
  customer_number STRING,
  first_name STRING,
  last_name STRING,
  country STRING,
  marital_status STRING,
  gender STRING,
  birthdate DATE,
  create_date DATE,
  attribute_hash STRING,
  effective_from DATE,
  effective_to DATE,
  is_current BOOLEAN,
  
  -- Quality metadata
  failure_reason STRING,
  failure_severity STRING,  -- CRITICAL | ERROR | WARNING
  quarantine_timestamp TIMESTAMP,
  remediation_status STRING,  -- PENDING | FIXED | IGNORED
  remediation_notes STRING
)
USING DELTA;

SELECT 'Customers quarantine table created' as status;

In [0]:
%sql
-- Quarantine table for dim_products failed quality checks
CREATE TABLE IF NOT EXISTS workspace.gold.dim_products_quarantine (
  -- Original product columns
  product_key BIGINT,
  product_id STRING,
  product_number STRING,
  product_name STRING,
  category_id STRING,
  category STRING,
  subcategory STRING,
  maintenance_flag STRING,
  product_line STRING,
  start_date DATE,
  attribute_hash STRING,
  effective_from DATE,
  effective_to DATE,
  is_current BOOLEAN,
  
  -- Quality metadata
  failure_reason STRING,
  failure_severity STRING,  -- CRITICAL | ERROR | WARNING
  quarantine_timestamp TIMESTAMP,
  remediation_status STRING,  -- PENDING | FIXED | IGNORED
  remediation_notes STRING
)
USING DELTA;

SELECT 'Products quarantine table created' as status;

In [0]:
%sql
-- Quarantine table for fact_sales failed quality checks
-- Note: Adjust columns based on actual fact_sales schema
CREATE TABLE IF NOT EXISTS workspace.gold.fact_sales_quarantine (
  -- Typical fact table columns (adjust as needed)
  sales_key BIGINT,
  order_date DATE,
  customer_key BIGINT,
  product_key BIGINT,
  quantity INT,
  unit_price DECIMAL(10,2),
  total_amount DECIMAL(10,2),
  
  -- Quality metadata
  failure_reason STRING,
  failure_severity STRING,  -- CRITICAL | ERROR | WARNING
  quarantine_timestamp TIMESTAMP,
  remediation_status STRING,  -- PENDING | FIXED | IGNORED
  remediation_notes STRING
)
USING DELTA;

SELECT 'Sales quarantine table created (adjust schema as needed)' as status;

In [0]:
%sql
-- Verify all data quality infrastructure is created
SELECT 
  'Metrics Table' as table_type,
  'gold.data_quality_metrics' as table_name,
  (SELECT COUNT(*) FROM workspace.gold.data_quality_metrics) as row_count
UNION ALL
SELECT 
  'Quarantine',
  'gold.dim_customers_quarantine',
  (SELECT COUNT(*) FROM workspace.gold.dim_customers_quarantine)
UNION ALL
SELECT 
  'Quarantine',
  'gold.dim_products_quarantine',
  (SELECT COUNT(*) FROM workspace.gold.dim_products_quarantine)
UNION ALL
SELECT 
  'Quarantine',
  'gold.fact_sales_quarantine',
  (SELECT COUNT(*) FROM workspace.gold.fact_sales_quarantine);

-- Show infrastructure is ready
SELECT '✅ Data Quality Infrastructure Ready!' as status;